# 01 Data Cleaning and Preprocessing

This notebook shows the complete preprocessing flow required by the project: raw data inspection, missing value handling, feature engineering, and standardization preview.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PROCESSED_DIR, RAW_DATASET_PATH
from src.data_loader import load_temperature_data
from src.feature_engineering import build_south_asia_subset
from src.preprocessing import build_modeling_dataset, clean_temperature_data

pd.set_option('display.max_columns', 50)

## Load Raw Dataset

In [2]:
raw_df = load_temperature_data(RAW_DATASET_PATH)
raw_df.head()

,dt,AverageTemperature,AverageTemperatureUncertainty,City,Country,Latitude,Longitude
0,1743-11-01,6.068,1.737,Århus,Denmark,57.05N,10.33E
1,1743-12-01,NaN,NaN,Århus,Denmark,57.05N,10.33E
2,1744-01-01,NaN,NaN,Århus,Denmark,57.05N,10.33E
3,1744-02-01,NaN,NaN,Århus,Denmark,57.05N,10.33E
4,1744-03-01,NaN,NaN,Århus,Denmark,57.05N,10.33E


In [3]:
print('Raw shape:', raw_df.shape)
print('Rows:', raw_df.shape[0])
print('Columns:', raw_df.shape[1])

Raw shape: (8599212, 7)
Rows: 8599212
Columns: 7


## Missing Values Before Cleaning

In [4]:
missing_before = raw_df.isna().sum().sort_values(ascending=False).reset_index()
missing_before.columns = ['Column', 'Missing Values']
missing_before

,Column,Missing Values
0,AverageTemperature,364130
1,AverageTemperatureUncertainty,364130
2,dt,0
3,City,0
4,Country,0
5,Latitude,0
6,Longitude,0


In [5]:
print('Duplicate rows before cleaning:', raw_df.duplicated().sum())

Duplicate rows before cleaning: 0


## Apply Cleaning and Feature Engineering

In [6]:
cleaned_df = clean_temperature_data(raw_df, min_year=1900)
cleaned_df.head()

,dt,AverageTemperature,AverageTemperatureUncertainty,City,Country,Latitude,Longitude,LatitudeValue,LongitudeValue,Year,Month,Quarter,Season,Hemisphere,Decade,YearsSince1900,RegionTag,Lag1Temperature,Lag12Temperature,Rolling12MeanTemperature,HistoricalCityMonthMean,TemperatureAnomaly,CityYearMeanTemperature,DecadeMeanTemperature,ClimateRiskIndex
0,1900-01-01,8.089,0.655,A Coruña,Spain,42.59N,8.73W,42.59,-8.73,1900,1,1,Winter,Northern,1900,0,Global,NaN,NaN,NaN,NaN,NaN,13.267917,17.346918,-0.744104
1,1900-02-01,9.772,1.227,A Coruña,Spain,42.59N,8.73W,42.59,-8.73,1900,2,1,Winter,Northern,1900,0,Global,8.089,NaN,NaN,NaN,NaN,13.267917,17.346918,1.364147
2,1900-03-01,8.018,1.127,A Coruña,Spain,42.59N,8.73W,42.59,-8.73,1900,3,1,Spring,Northern,1900,0,Global,9.772,NaN,NaN,NaN,NaN,13.267917,17.346918,0.995572
3,1900-04-01,12.596,0.608,A Coruña,Spain,42.59N,8.73W,42.59,-8.73,1900,4,2,Spring,Northern,1900,0,Global,8.018,NaN,NaN,NaN,NaN,13.267917,17.346918,-0.917335
4,1900-05-01,13.057,0.535,A Coruña,Spain,42.59N,8.73W,42.59,-8.73,1900,5,2,Spring,Northern,1900,0,Global,12.596,NaN,NaN,NaN,NaN,13.267917,17.346918,-1.186395


In [7]:
print('Cleaned shape:', cleaned_df.shape)
print('Rows removed during cleaning:', raw_df.shape[0] - cleaned_df.shape[0])
print('Date range:', cleaned_df['dt'].min(), 'to', cleaned_df['dt'].max())

Cleaned shape: (4788080, 25)
Rows removed during cleaning: 3811132
Date range: 1900-01-01 00:00:00 to 2013-09-01 00:00:00


## Missing Values After Cleaning

In [8]:
missing_after = cleaned_df.isna().sum().sort_values(ascending=False).reset_index()
missing_after.columns = ['Column', 'Missing Values']
missing_after.head(15)

,Column,Missing Values
0,HistoricalCityMonthMean,41376
1,TemperatureAnomaly,41376
2,Lag12Temperature,41376
3,Rolling12MeanTemperature,20688
4,Lag1Temperature,3448
5,AverageTemperature,0
6,AverageTemperatureUncertainty,0
7,City,0
8,dt,0
9,LongitudeValue,0


## Basic Cleaning Summary

In [9]:
summary_df = pd.DataFrame({
    'Step': [
        'Original dataset',
        'Rows after removing missing target/date and duplicates',
        'Rows after restricting to year 1900 onward'
    ],
    'Rows': [
        len(raw_df),
        len(raw_df.dropna(subset=['dt', 'AverageTemperature']).drop_duplicates()),
        len(cleaned_df)
    ]
})
summary_df

,Step,Rows
0,Original dataset,8599212
1,Rows after removing missing target/date and du...,8235082
2,Rows after restricting to year 1900 onward,4788080


## Feature Engineering Output

In [10]:
engineered_columns = [
    'dt', 'AverageTemperature', 'City', 'Country', 'Year', 'Month', 'Quarter',
    'Season', 'Hemisphere', 'Decade', 'RegionTag', 'LatitudeValue', 'LongitudeValue',
    'Lag1Temperature', 'Lag12Temperature', 'Rolling12MeanTemperature',
    'HistoricalCityMonthMean', 'TemperatureAnomaly', 'ClimateRiskIndex'
]
cleaned_df[engineered_columns].head(10)

,dt,AverageTemperature,City,Country,Year,Month,Quarter,Season,Hemisphere,Decade,RegionTag,LatitudeValue,LongitudeValue,Lag1Temperature,Lag12Temperature,Rolling12MeanTemperature,HistoricalCityMonthMean,TemperatureAnomaly,ClimateRiskIndex
0,1900-01-01,8.089000,A Coruña,Spain,1900,1,1,Winter,Northern,1900,Global,42.59,-8.73,NaN,NaN,NaN,NaN,NaN,-0.744104
1,1900-02-01,9.772000,A Coruña,Spain,1900,2,1,Winter,Northern,1900,Global,42.59,-8.73,8.089000,NaN,NaN,NaN,NaN,1.364147
2,1900-03-01,8.018000,A Coruña,Spain,1900,3,1,Spring,Northern,1900,Global,42.59,-8.73,9.772000,NaN,NaN,NaN,NaN,0.995572
3,1900-04-01,12.596000,A Coruña,Spain,1900,4,2,Spring,Northern,1900,Global,42.59,-8.73,8.018000,NaN,NaN,NaN,NaN,-0.917335
4,1900-05-01,13.057000,A Coruña,Spain,1900,5,2,Spring,Northern,1900,Global,42.59,-8.73,12.596000,NaN,NaN,NaN,NaN,-1.186395
5,1900-06-01,16.177000,A Coruña,Spain,1900,6,2,Summer,Northern,1900,Global,42.59,-8.73,13.057000,NaN,NaN,NaN,NaN,-0.773590
6,1900-07-01,19.721001,A Coruña,Spain,1900,7,3,Summer,Northern,1900,Global,42.59,-8.73,16.177000,NaN,11.284833,NaN,NaN,-0.534016
7,1900-08-01,18.222000,A Coruña,Spain,1900,8,3,Summer,Northern,1900,Global,42.59,-8.73,19.721001,NaN,12.490000,NaN,NaN,-0.423444
8,1900-09-01,18.306999,A Coruña,Spain,1900,9,3,Autumn,Northern,1900,Global,42.59,-8.73,18.222000,NaN,13.206500,NaN,NaN,1.570549
9,1900-10-01,14.667000,A Coruña,Spain,1900,10,4,Autumn,Northern,1900,Global,42.59,-8.73,18.306999,NaN,13.773222,NaN,NaN,-0.780962


## Standardization Preview

The project standardizes numeric features during model training using `StandardScaler`. This cell shows a small example so the preprocessing requirement is clearly visible in the notebook.

In [11]:
numeric_features = [
    'Year', 'Month', 'Quarter', 'YearsSince1900', 'LatitudeValue', 'LongitudeValue',
    'AverageTemperatureUncertainty', 'HistoricalCityMonthMean', 'Lag1Temperature',
    'Lag12Temperature', 'Rolling12MeanTemperature'
]
standardization_sample = cleaned_df[numeric_features].dropna().sample(n=5, random_state=42)
scaler = StandardScaler()
scaled_values = scaler.fit_transform(standardization_sample)
scaled_df = pd.DataFrame(scaled_values, columns=numeric_features)
scaled_df

,Year,Month,Quarter,YearsSince1900,LatitudeValue,LongitudeValue,AverageTemperatureUncertainty,HistoricalCityMonthMean,Lag1Temperature,Lag12Temperature,Rolling12MeanTemperature
0,1.137541,-0.987829,-0.790569,1.137541,0.589529,-0.544004,-0.911996,1.303353,1.362393,0.846246,0.123173
1,-1.121405,0.808224,0.790569,-1.121405,0.589529,1.085579,1.874026,-0.248511,-0.228556,0.187828,-0.154013
2,0.613143,0.808224,0.790569,0.613143,-0.090588,-0.873285,-0.612546,-0.894139,-0.448303,-1.138650,-0.400010
3,0.653481,-1.436842,-1.581139,0.653481,0.816706,1.329456,0.156927,-1.181967,-1.495089,-1.161508,-1.318897
4,-1.282759,0.808224,0.790569,-1.282759,-1.905176,-0.997746,-0.506412,1.021264,0.809556,1.266083,1.749746


## Build South Asia Subset and Modeling Dataset

In [12]:
south_asia_df = build_south_asia_subset(cleaned_df)
modeling_df = build_modeling_dataset(cleaned_df)

print('South Asia subset shape:', south_asia_df.shape)
print('Modeling dataset shape:', modeling_df.shape)

South Asia subset shape: (672452, 25)
Modeling dataset shape: (75000, 25)


In [13]:
modeling_df[['dt', 'City', 'Country', 'AverageTemperature', 'HistoricalCityMonthMean', 'Lag12Temperature']].head()

,dt,City,Country,AverageTemperature,HistoricalCityMonthMean,Lag12Temperature
0,1900-07-01,Cartagena,Spain,23.469999,29.283001,12.283000
1,1900-08-01,Colombo,Sri Lanka,26.240000,15.205000,26.290001
2,1900-09-01,Colombo,Sri Lanka,25.975000,16.275999,27.888000
3,1900-10-01,Springfield,United States,11.560000,16.729000,18.907000
4,1901-01-01,Taunggyi,Burma,19.306000,19.009001,19.009001


## Save Processed Outputs

In [14]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

cleaned_path = PROCESSED_DIR / 'cleaned_temperature_data.csv'
south_asia_path = PROCESSED_DIR / 'south_asia_subset.csv'
modeling_path = PROCESSED_DIR / 'modeling_dataset.csv'

cleaned_df.to_csv(cleaned_path, index=False)
south_asia_df.to_csv(south_asia_path, index=False)
modeling_df.to_csv(modeling_path, index=False)

print(cleaned_path)
print(south_asia_path)
print(modeling_path)

D:\6th semester\DS\climate change predications\data\processed\cleaned_temperature_data.csv
D:\6th semester\DS\climate change predications\data\processed\south_asia_subset.csv
D:\6th semester\DS\climate change predications\data\processed\modeling_dataset.csv
